# Create a Content Understanding Business Card Analyzer

Use this notebook to create the same business card analyzer as `create-analyzer.py`, one step at a time. The notebook reads settings from the shared `workshop/.env` file and uses `DefaultAzureCredential` when `CONTENT_UNDERSTANDING_KEY` is empty.

## 1. Confirm Prerequisites

Before running the code cells, confirm that `workshop/.env` includes `CONTENT_UNDERSTANDING_ENDPOINT`, `CONTENT_UNDERSTANDING_KEY` if you want key-based authentication, and `ANALYZER_NAME`. If the key is empty, run `az login` first so `DefaultAzureCredential` can use your current Azure account.

In [1]:
from pathlib import Path
import json
import os

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

## 2. Locate the Notebook Files

This cell resolves the notebook folder, the shared environment file, and the business card analyzer schema. It works whether you open the notebook from VS Code or run it from the repository root.

In [2]:
notebook_dir = Path.cwd()
if notebook_dir.name != "02-create-analyzers-python-sdk":
    notebook_dir = Path("workshop/lab-02-content-understanding/02-create-analyzers-python-sdk").resolve()

env_path = notebook_dir.parents[1] / ".env"
schema_path = notebook_dir / "biz-card.json"

print(f"Notebook folder: {notebook_dir}")
print(f"Environment file: {env_path}")
print(f"Schema file: {schema_path}")

Notebook folder: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk
Environment file: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Schema file: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk\biz-card.json


## 3. Define Configuration Helpers

These helper functions match the script behavior: placeholder values such as `<your-content-understanding-endpoint>` are treated as empty, and the key is optional.

In [3]:
def optional_setting(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    return "" if not value or value.startswith("<") else value


def require_setting(name: str) -> str:
    value = optional_setting(name)
    if not value:
        raise ValueError(f"Set {name} in workshop/.env")
    return value


def get_credential():
    key = optional_setting("CONTENT_UNDERSTANDING_KEY")
    return AzureKeyCredential(key) if key else DefaultAzureCredential()

## 4. Load Environment Settings

Load the Content Understanding endpoint and analyzer name from the shared environment file.

In [4]:
load_dotenv(env_path, override=True)

ai_svc_endpoint = require_setting("CONTENT_UNDERSTANDING_ENDPOINT")
analyzer = require_setting("ANALYZER_NAME")
auth_mode = "AzureKeyCredential" if optional_setting("CONTENT_UNDERSTANDING_KEY") else "DefaultAzureCredential"

print(f"Endpoint: {ai_svc_endpoint}")
print(f"Analyzer name: {analyzer}")
print(f"Authentication mode: {auth_mode}")

Endpoint: https://4ublv6yuwkfh4-aifoundry.services.ai.azure.com/
Analyzer name: bizcardanalyzer
Authentication mode: DefaultAzureCredential


## 5. Load and Review the Business Card Schema

The schema defines the analyzer and the fields it should extract from business card images.

In [5]:
with schema_path.open("r", encoding="utf-8") as file:
    schema_json = json.load(file)

field_names = list(schema_json.get("fieldSchema", {}).get("fields", {}).keys())
print(f"Schema description: {schema_json.get('description')}")
print(f"Fields: {', '.join(field_names)}")
schema_json

Schema description: Business card analyzer
Fields: Company, Name, Title, Email, Phone


{'description': 'Business card analyzer',
 'baseAnalyzerId': 'prebuilt-document',
 'models': {'completion': 'gpt-4.1', 'embedding': 'text-embedding-3-large'},
 'config': {'returnDetails': True},
 'fieldSchema': {'fields': {'Company': {'type': 'string',
    'method': 'extract',
    'description': 'Company or organization'},
   'Name': {'type': 'string',
    'method': 'extract',
    'description': "Contact's name"},
   'Title': {'type': 'string',
    'method': 'extract',
    'description': "Contact's job title"},
   'Email': {'type': 'string',
    'method': 'extract',
    'description': "Contact's email address"},
   'Phone': {'type': 'string',
    'method': 'extract',
    'description': "Contact's phone number"}}}}

## 6. Create the Content Understanding Client

Create the SDK client with the selected credential.

In [6]:
client = ContentUnderstandingClient(
    endpoint=ai_svc_endpoint,
    credential=get_credential()
)

client

## 7. Create or Replace the Analyzer

This starts a long-running operation. Re-running the cell replaces the existing analyzer with the same `ANALYZER_NAME` because `allow_replace=True` is set.

In [7]:
print(f"Creating {analyzer}")

poller = client.begin_create_analyzer(
    analyzer_id=analyzer,
    resource=schema_json,
    allow_replace=True
)

result = poller.result()
print(f"Analyzer '{analyzer}' created successfully.")
print(f"Status: {result['status'] if isinstance(result, dict) else 'Succeeded'}")

Creating bizcardanalyzer
Analyzer 'bizcardanalyzer' created successfully.
Status: Succeeded


## 8. Next Step

After the analyzer is created, run `read-card.py` or create a separate analysis notebook to submit `biz-card-1.png` and `biz-card-2.png` to the analyzer.